In [1]:
import pandas as pd
import numpy as np
from sklearn import datasets
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
%matplotlib inline

# Read Data

In [3]:
df = pd.read_csv("../data/cc_ungrouped.csv")

In [4]:
# check dimensionality
print("Number of rows:", df.shape[0])
print("Number of features/columns:", df.shape[1])

Number of rows: 713832
Number of features/columns: 43


In [5]:
df.head(5)

,Name,Address,State,District,lat,lng,IDPatient,IDAppointment,ICD-10,DiagnosisDescription,...,PatientTotal,Total_Panel,CashPatient,PanelPatient,Countries/Geography,Ethnicities,Religions,Social Status,UpdatedGender,Age
0,Clinic A,"No 13, Jalan Tengku Ampuan Zabedah B 9/B, Seks...",Selangor,Shah Alam,3.081702,101.527401,1000391,37188523,NaN,NaN,...,0.0,351.0,No,Yes,Malaysia,Malay,Islam,Married,M,54.0
1,Clinic A,"No 13, Jalan Tengku Ampuan Zabedah B 9/B, Seks...",Selangor,Shah Alam,3.081702,101.527401,1000391,44240054,NaN,NaN,...,0.0,299.0,No,Yes,Malaysia,Malay,Islam,Married,M,54.0
2,Clinic A,"No 13, Jalan Tengku Ampuan Zabedah B 9/B, Seks...",Selangor,Shah Alam,3.081702,101.527401,1000880,14071827,NaN,NaN,...,150.0,0.0,Yes,No,Malaysia,Chinese,Buddhist,Married,F,65.0
3,Clinic A,"No 13, Jalan Tengku Ampuan Zabedah B 9/B, Seks...",Selangor,Shah Alam,3.081702,101.527401,1000880,37850933,NaN,NaN,...,90.0,0.0,Yes,No,Malaysia,Chinese,Buddhist,Married,F,65.0
4,Clinic A,"No 13, Jalan Tengku Ampuan Zabedah B 9/B, Seks...",Selangor,Shah Alam,3.081702,101.527401,1002889,13549403,NaN,NaN,...,100.0,0.0,Yes,No,Malaysia,Malay,Islam,Single person,M,29.0


## Select Columns

In [6]:
cols_to_use = [
    'Name', 'Address', 'State', 'District',
    'IDPatient', 'IDAppointment', 'ICD-10', 'DiagnosisDescription',
    'DateTime',
    'NewPatient',
    'RegistrationTime(Mins)', 'WaitingTime(Mins)', 'VisitTime(Mins)',
    'StaffRole', 'StaffName',
    'UpdatedGender', 'Age',
    'Countries/Geography', 'Ethnicities', 
    'Religions', 'Social Status'#, 'PatientCount'
]

df_cleaned = df[cols_to_use].copy()

## Checking data types (attribute types)

In [7]:
df_cleaned.dtypes

Name                       object
Address                    object
State                      object
District                   object
IDPatient                   int64
IDAppointment               int64
ICD-10                     object
DiagnosisDescription       object
DateTime                   object
NewPatient                 object
RegistrationTime(Mins)    float64
WaitingTime(Mins)         float64
VisitTime(Mins)           float64
StaffRole                  object
StaffName                  object
UpdatedGender              object
Age                       float64
Countries/Geography        object
Ethnicities                object
Religions                  object
Social Status              object
dtype: object

In [8]:
# Convert 'DateTime' column to proper datetime objects
df_cleaned['DateTime'] = pd.to_datetime(df_cleaned['DateTime'])

# Convert IDs to Strings 
# Convert these to strings so the machine learning model treats them as 
# unique labels (like a name tag) rather than numbers to be added or multiplied.
df_cleaned['IDPatient'] = df_cleaned['IDPatient'].astype(str)
df_cleaned['IDAppointment'] = df_cleaned['IDAppointment'].astype(str)

# Extract useful time components for analysis
df_cleaned['Date'] = df_cleaned['DateTime'].dt.date
df_cleaned['Hour'] = df_cleaned['DateTime'].dt.hour
df_cleaned['DayOfWeek'] = df_cleaned['DateTime'].dt.dayofweek
df_cleaned['Month'] = df_cleaned['DateTime'].dt.month

In [9]:
df_cleaned.dtypes

Name                              object
Address                           object
State                             object
District                          object
IDPatient                         object
IDAppointment                     object
ICD-10                            object
DiagnosisDescription              object
DateTime                  datetime64[ns]
NewPatient                        object
RegistrationTime(Mins)           float64
WaitingTime(Mins)                float64
VisitTime(Mins)                  float64
StaffRole                         object
StaffName                         object
UpdatedGender                     object
Age                              float64
Countries/Geography               object
Ethnicities                       object
Religions                         object
Social Status                     object
Date                              object
Hour                               int32
DayOfWeek                          int32
Month           

# Missing data

In [10]:
df_cleaned.describe()

,DateTime,RegistrationTime(Mins),WaitingTime(Mins),VisitTime(Mins),Age,Hour,DayOfWeek,Month
count,713832,713832.000000,713832.000000,713832.000000,713742.000000,713832.000000,713832.00000,713832.000000
mean,2025-08-01 16:40:50.087686912,60.290915,5.498138,8.792131,34.656031,14.412467,2.72432,7.931915
min,2024-03-28 09:11:22,-12989.483333,-521.866667,-0.016667,0.000000,0.000000,0.00000,1.000000
25%,2025-05-14 05:40:19.500000,1.383333,0.000000,1.200000,23.000000,11.000000,1.00000,6.000000
50%,2025-09-03 10:20:57,4.633333,0.000000,4.900000,34.000000,14.000000,3.00000,9.000000
75%,2025-11-19 09:36:24.500000,15.716667,2.400000,10.050000,46.000000,18.000000,4.00000,11.000000
max,2026-01-14 01:51:03,200307.333333,65396.916667,49806.583333,142.000000,23.000000,6.00000,12.000000
std,NaN,1100.925234,156.840800,110.211086,18.504010,4.689815,1.98562,3.491657


In [11]:
df_cleaned.isna().sum()

Name                           0
Address                        0
State                          0
District                       0
IDPatient                      0
IDAppointment                  0
ICD-10                    703462
DiagnosisDescription      703499
DateTime                       0
NewPatient                     0
RegistrationTime(Mins)         0
WaitingTime(Mins)              0
VisitTime(Mins)                0
StaffRole                      0
StaffName                 292877
UpdatedGender               1777
Age                           90
Countries/Geography         5442
Ethnicities               593886
Religions                 323748
Social Status             617358
Date                           0
Hour                           0
DayOfWeek                      0
Month                          0
dtype: int64

## Fill NA

* **ICD-10 & DiagnosisDescription:** These fields contain a high proportion of missing values. To avoid introducing noise or assumptions, these columns were removed from the dataset.
* **StaffName:** We don't know who the doctor was, so label them as "Unknown".
* **UpdatedGender:** Since this is a small number missing, we will fill it with the most common gender (Mode).
* **Age:** A very small number missing. Fill it with the median age (safe from outliers).
* **Countries/Geography:** Contains a very low number of missing values (approx. 5k). Filled with the most frequent geography (Mode).
* **Ethnicities, Religions, & Social Status:** These variables contain massive missing gaps (300k - 600k+ missing rows). To prevent severe imputation bias in the K-Modes clustering algorithm, these missing values are categorized as "Not Disclosed" rather than imputed or dropped.

In [12]:
# 1. Drop diagnosis-related columns due to high missing values
df_cleaned = df_cleaned.drop(columns=['ICD-10', 'DiagnosisDescription'])

# 2. Fill missing Staff Names and Roles (kept for future safety)
df_cleaned['StaffName'] = df_cleaned['StaffName'].fillna('Unknown')
if 'StaffRole' in df_cleaned.columns:
    df_cleaned['StaffRole'] = df_cleaned['StaffRole'].fillna('Unknown')

# 3. Fill missing Gender with the most frequent value (Mode)
gender_mode = df_cleaned['UpdatedGender'].mode()[0]
df_cleaned['UpdatedGender'] = df_cleaned['UpdatedGender'].fillna(gender_mode)

# 4. Fill missing Age with the Median
age_median = df_cleaned['Age'].median()
df_cleaned['Age'] = df_cleaned['Age'].fillna(age_median)

# 5. Fill Countries/Geography with the Mode (Low missing count)
country_mode = df_cleaned['Countries/Geography'].mode()[0]
df_cleaned['Countries/Geography'] = df_cleaned['Countries/Geography'].fillna(country_mode)

# 6. Fill high-missing demographics with "Not Disclosed" to avoid bias
high_missing_cols = ['Ethnicities', 'Religions', 'Social Status']
for col in high_missing_cols:
    df_cleaned[col] = df_cleaned[col].fillna('Not Disclosed')


## Check for NA after filling missing value

In [13]:
print("Missing values after cleaning:")
print(df_cleaned.isna().sum())

Missing values after cleaning:
Name                      0
Address                   0
State                     0
District                  0
IDPatient                 0
IDAppointment             0
DateTime                  0
NewPatient                0
RegistrationTime(Mins)    0
WaitingTime(Mins)         0
VisitTime(Mins)           0
StaffRole                 0
StaffName                 0
UpdatedGender             0
Age                       0
Countries/Geography       0
Ethnicities               0
Religions                 0
Social Status             0
Date                      0
Hour                      0
DayOfWeek                 0
Month                     0
dtype: int64


# Check for Duplicated Data

In [14]:
print("Total duplicated rows: ", sum(df_cleaned.duplicated()))

Total duplicated rows:  0


# Data Cleaning
Clean outliers and invalid data

In [15]:
time_cols = ['RegistrationTime(Mins)', 'WaitingTime(Mins)', 'VisitTime(Mins)']

# Check for negative values
for col in time_cols:
    invalid_count = df_cleaned[df_cleaned[col] < 0].shape[0]
    print(f"Removing {invalid_count} rows with negative {col}...")
    df_cleaned = df_cleaned[df_cleaned[col] >= 0]

# Check for extreme values (e.g., > 24 hours [1440 mins])
for col in time_cols:
    extreme_count = df_cleaned[df_cleaned[col] > 1440].shape[0]
    print(f"Removing {extreme_count} rows with {col} > 24 hours...")
    df_cleaned = df_cleaned[df_cleaned[col] <= 1440]

print("\nFinal Data Shape after Outlier Removal:")
print(df_cleaned.shape)
print(df_cleaned.describe()) # Check 'max' and 'min' rows to ensure they look sane

Removing 268 rows with negative RegistrationTime(Mins)...
Removing 662 rows with negative WaitingTime(Mins)...
Removing 1 rows with negative VisitTime(Mins)...
Removing 3114 rows with RegistrationTime(Mins) > 24 hours...
Removing 21 rows with WaitingTime(Mins) > 24 hours...
Removing 34 rows with VisitTime(Mins) > 24 hours...

Final Data Shape after Outlier Removal:
(709732, 23)
                            DateTime  RegistrationTime(Mins)  \
count                         709732           709732.000000   
mean   2025-08-02 00:01:02.983907584               21.061140   
min              2024-03-28 09:11:22                0.000000   
25%    2025-05-14 16:54:37.249999872                1.383333   
50%              2025-09-03 18:01:40                4.600000   
75%    2025-11-19 11:31:21.249999872               15.450000   
max              2026-01-14 01:51:03             1439.916667   
std                              NaN               67.658285   

       WaitingTime(Mins)  VisitTime(Mins) 

## Refined Cleaning
Cap Registration Time at 120 minutes (2 hours). This covers 99% of valid cases. Anything longer is likely a system error (e.g., someone forgot to "stop" the timer).

In [16]:
# Filter Registration Time to realistic limits (< 120 mins)
print(f"Before refining: {df_cleaned.shape}")
df_cleaned = df_cleaned[df_cleaned['RegistrationTime(Mins)'] <= 120]

# Filter Age to realistic limits (< 100 years)
df_cleaned = df_cleaned[df_cleaned['Age'] <= 100]
print(f"After refining: {df_cleaned.shape}")

Before refining: (709732, 23)
After refining: (686112, 23)


In [17]:
# Check for suitable min/max waiting time and visit time
percentiles_to_show = [0.01, 0.25, 0.50, 0.75, 0.99, 0.999]

stats = df_cleaned[['WaitingTime(Mins)', 'VisitTime(Mins)']].describe(percentiles=percentiles_to_show)

print(stats)

       WaitingTime(Mins)  VisitTime(Mins)
count      686112.000000    686112.000000
mean            5.154233         7.996567
std            15.726592        11.496063
min             0.000000         0.000000
1%              0.000000         0.000000
25%             0.000000         1.383333
50%             0.000000         5.116667
75%             2.800000        10.266667
99%            61.283333        49.783333
99.9%         138.598150       105.874083
max          1336.033333      1399.483333


Cap Waiting Time and Visit Time to 99.9% of the upper limit to remove outliers.

For Visit Time, the bottom 1% is still 0 minutes. Since 0-minute visits are impossible, I added a safety line to ensure the minimum is at least 1 minute, even if the 1% statistic is lower.

In [18]:
# Define Percentiles
UPPER_QUANTILE = 0.999 
LOWER_QUANTILE = 0.01

# Calculate Thresholds
wait_max_cap = df_cleaned['WaitingTime(Mins)'].quantile(UPPER_QUANTILE)
visit_max_cap = df_cleaned['VisitTime(Mins)'].quantile(UPPER_QUANTILE)
visit_min_cap = df_cleaned['VisitTime(Mins)'].quantile(LOWER_QUANTILE)

print(f"Upper Cap for Waiting Time ({UPPER_QUANTILE*100}%): {wait_max_cap:.2f} mins")
print(f"Upper Cap for Visit Time ({UPPER_QUANTILE*100}%): {visit_max_cap:.2f} mins")
print(f"Lower Cap for Visit Time ({LOWER_QUANTILE*100}%): {visit_min_cap:.2f} mins")

# SAFETY CHECK: Ensure Minimum Visit Time is Logical
# Since the 1% value might still be 0.0, I force a logical minimum of 1 minute.
if visit_min_cap < 1.0:
    print("Note: 1% quantile is too low (0 mins). Using 1.0 mins as the logical minimum.")
    visit_min_cap = 1.0

# Cap the Maximums (Winsorization)
df_cleaned.loc[df_cleaned['WaitingTime(Mins)'] > wait_max_cap, 'WaitingTime(Mins)'] = wait_max_cap
df_cleaned.loc[df_cleaned['VisitTime(Mins)'] > visit_max_cap, 'VisitTime(Mins)'] = visit_max_cap

# Cap the Minimum (Flooring)
df_cleaned.loc[df_cleaned['VisitTime(Mins)'] < visit_min_cap, 'VisitTime(Mins)'] = visit_min_cap

# Verify Results
print("\nFinal Range Check:")
print(df_cleaned[['WaitingTime(Mins)', 'VisitTime(Mins)']].describe())

Upper Cap for Waiting Time (99.9%): 138.60 mins
Upper Cap for Visit Time (99.9%): 105.87 mins
Lower Cap for Visit Time (1.0%): 0.00 mins
Note: 1% quantile is too low (0 mins). Using 1.0 mins as the logical minimum.

Final Range Check:
       WaitingTime(Mins)  VisitTime(Mins)
count      686112.000000    686112.000000
mean            5.038450         8.114759
std            12.886043        10.174020
min             0.000000         1.000000
25%             0.000000         1.383333
50%             0.000000         5.116667
75%             2.800000        10.266667
max           138.598150       105.874083


Remove the last date because it may be incomplete (cutoff time 01:51 AM)

In [19]:
max_date = df_cleaned['Date'].max()
print(f"Removing incomplete final date: {max_date}")

df_cleaned = df_cleaned[df_cleaned['Date'] < max_date]

Removing incomplete final date: 2026-01-14


# Feature Engineering
## Patient Arrival Modelling
This section creates new features for modeling. I extract time-based fields (hour, day, month, weekend), mark whether each date is a public holiday, calculate hourly patient demand, convert categorical text fields into numeric labels, and scale numerical columns for consistency before training the model.

## Temporal Features (Time & Holidays)
This section creates time-based features (date, hour, weekday, weekend) and adds a detailed state-specific public holiday indicator. A dictionary of 2024–2025 holidays is used, and each row is checked to determine whether its date is a holiday for that specific state. The result is stored as a binary feature IsPublicHoliday.

In [20]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
# Create Basic Time Features
df_cleaned['Date'] = df_cleaned['DateTime'].dt.date
df_cleaned['Hour'] = df_cleaned['DateTime'].dt.hour
df_cleaned['DayOfWeek'] = df_cleaned['DateTime'].dt.dayofweek # 0=Monday, 6=Sunday
df_cleaned['Month'] = df_cleaned['DateTime'].dt.month
df_cleaned['IsWeekend'] = df_cleaned['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

#  Public Holidays
holiday_data = {
    # --- 2024 ---
    '2024-01-01': ['National_Except', ['Johor', 'Kedah', 'Kelantan', 'Perlis', 'Terengganu']],
    '2024-01-14': ['Negeri Sembilan'],
    '2024-01-25': ['Johor', 'Kuala Lumpur', 'Negeri Sembilan', 'Pulau Pinang', 'Perak', 'Putrajaya', 'Selangor'], # 'Penang' -> 'Pulau Pinang' to match dataset likely
    '2024-02-01': ['Kuala Lumpur', 'Labuan', 'Putrajaya'],
    '2024-02-08': ['Kedah', 'Negeri Sembilan', 'Perlis', 'Terengganu'],
    '2024-02-10': ['National'],
    '2024-02-11': ['National'],
    '2024-02-12': ['National_Except', ['Johor', 'Kedah', 'Kelantan', 'Terengganu']],
    '2024-02-20': ['Melaka'],
    '2024-03-04': ['Terengganu'],
    '2024-03-12': ['Johor', 'Kedah', 'Melaka'],
    '2024-03-23': ['Johor'],
    '2024-03-28': ['National_Except', ['Johor', 'Kedah', 'Melaka', 'Negeri Sembilan', 'Sabah', 'Sarawak']],
    '2024-03-29': ['Sabah', 'Sarawak'],
    '2024-04-10': ['National'],
    '2024-04-11': ['National'],
    '2024-04-26': ['Terengganu'],
    '2024-05-01': ['National'],
    '2024-05-17': ['Perlis'],
    '2024-05-22': ['Pahang', 'National'], # Both Hol Pahang and Wesak Day. 'National' covers it.
    '2024-05-30': ['Labuan', 'Sabah'],
    '2024-05-31': ['Labuan', 'Sabah'],
    '2024-06-01': ['Sarawak'],
    '2024-06-02': ['Sarawak'],
    '2024-06-03': ['Sarawak', 'National'], # Agong's Bday covers National
    '2024-06-16': ['Terengganu'],
    '2024-06-17': ['National'],
    '2024-06-18': ['Kedah', 'Kelantan', 'Perlis', 'Terengganu'],
    '2024-06-30': ['Kedah'],
    '2024-07-07': ['Pulau Pinang', 'National'], # Awal Muharram covers National
    '2024-07-13': ['Pulau Pinang'],
    '2024-07-22': ['Sarawak'],
    '2024-07-30': ['Pahang'],
    '2024-08-11': ['Johor'],
    '2024-08-24': ['Melaka'],
    '2024-08-31': ['National'],
    '2024-09-16': ['National'], # Malaysia Day + Prophet Bday
    '2024-09-29': ['Kelantan'],
    '2024-09-30': ['Kelantan'],
    '2024-10-05': ['Sabah'],
    '2024-10-12': ['Sarawak'],
    '2024-10-31': ['National_Except', ['Sarawak']],
    '2024-11-01': ['Perak'],
    '2024-12-11': ['Selangor'],
    '2024-12-24': ['Sabah'],
    '2024-12-25': ['National'],

    # --- 2025 ---
    '2025-01-01': ['National_Except', ['Johor', 'Kedah', 'Kelantan', 'Perlis', 'Terengganu']],
    '2025-01-14': ['Negeri Sembilan'],
    '2025-01-27': ['Kedah', 'Negeri Sembilan', 'Perlis', 'Terengganu'],
    '2025-01-29': ['National'],
    '2025-01-30': ['National'],
    '2025-02-01': ['Kuala Lumpur', 'Labuan', 'Putrajaya'],
    '2025-02-11': ['Johor', 'Kedah', 'Kuala Lumpur', 'Negeri Sembilan', 'Pulau Pinang', 'Perak', 'Putrajaya', 'Selangor'],
    '2025-02-20': ['Melaka'],
    '2025-03-02': ['Johor', 'Kedah'],
    '2025-03-03': ['Johor'],
    '2025-03-04': ['Terengganu'],
    '2025-03-18': ['National_Except', ['Johor', 'Kedah', 'Melaka', 'Negeri Sembilan', 'Sabah', 'Sarawak']],
    '2025-03-23': ['Johor'],
    '2025-03-24': ['Johor'],
    '2025-03-30': ['Kedah', 'Kelantan', 'Terengganu', 'Sabah'],
    '2025-03-31': ['National'],
    '2025-04-01': ['National'],
    '2025-04-02': ['Melaka'],
    '2025-04-18': ['Sabah', 'Sarawak'],
    '2025-04-26': ['Terengganu'],
    '2025-04-27': ['Terengganu'],
    '2025-04-28': ['Johor'],
    '2025-05-01': ['National'],
    '2025-05-12': ['National'],
    '2025-05-17': ['Perlis'],
    '2025-05-22': ['Pahang'],
    '2025-05-30': ['Labuan', 'Sabah'],
    '2025-05-31': ['Labuan', 'Sabah'],
    '2025-06-01': ['Sarawak'],
    '2025-06-02': ['Sarawak', 'National'],
    '2025-06-03': ['Sarawak'],
    '2025-06-06': ['Kelantan', 'Terengganu'],
    '2025-06-07': ['National'],
    '2025-06-08': ['Kedah', 'Kelantan', 'Perlis', 'Terengganu'],
    '2025-06-09': ['Kelantan', 'Perlis', 'Terengganu'],
    '2025-06-22': ['Kedah'],
    '2025-06-27': ['National'],
    '2025-06-29': ['Kedah'],
    '2025-07-07': ['Pulau Pinang'],
    '2025-07-12': ['Pulau Pinang'],
    '2025-07-22': ['Sarawak'],
    '2025-07-30': ['Pahang'],
    '2025-07-31': ['Johor'],
    '2025-08-24': ['Melaka'],
    '2025-08-25': ['Melaka'],
    '2025-08-31': ['National'],
    '2025-09-01': ['National_Except', ['Kedah', 'Kelantan', 'Terengganu']],
    '2025-09-05': ['National'],
    '2025-09-07': ['Kedah'],
    '2025-09-15': ['National'],
    '2025-09-16': ['National'],
    '2025-09-29': ['Kelantan'],
    '2025-09-30': ['Kelantan'],
    '2025-10-11': ['Sarawak'],
    '2025-10-20': ['National_Except', ['Sarawak']],
    '2025-11-07': ['Perak'],
    '2025-12-11': ['Selangor'],
    '2025-12-24': ['Sabah'],
    '2025-12-25': ['National']
}

# FUNCTION TO CHECK HOLIDAY
def is_holiday(row):
    """
    Checks if the date is a holiday for the specific state in that row.
    """
    date_str = str(row['Date'])
    state = str(row['State'])
    
    # If date is not in our holiday list, it's not a holiday
    if date_str not in holiday_data:
        return 0
    
    rule = holiday_data[date_str]
    
    # Case 1: National Holiday (All states)
    if rule == ['National'] or 'National' in rule:
        return 1
    
    # Case 2: National Except specific states
    if rule[0] == 'National_Except':
        exceptions = rule[1]
        if state in exceptions:
            return 0 # It is NOT a holiday for this exception state
        else:
            return 1 # It IS a holiday for everyone else
            
    # Case 3: Specific State List
    # (e.g., ['Selangor', 'Perak'])
    if state in rule:
        return 1
        
    return 0

# Ensure Date is string for dictionary lookup
df_cleaned['Date_Str'] = df_cleaned['Date'].astype(str)

# Apply the function row-by-row
df_cleaned['IsPublicHoliday'] = df_cleaned.apply(is_holiday, axis=1)

# Drop the temporary string column
df_cleaned.drop(columns=['Date_Str'], inplace=True)

print("Enhanced State-Specific Holiday Feature Added.")
print(df_cleaned[['Date', 'State', 'IsPublicHoliday']].head(10))
print(df_cleaned["IsPublicHoliday"].value_counts())

Enhanced State-Specific Holiday Feature Added.
         Date     State  IsPublicHoliday
0  2025-09-15  Selangor                1
1  2025-10-05  Selangor                0
2  2025-06-26  Selangor                0
3  2025-11-09  Selangor                0
4  2025-05-06  Selangor                0
5  2025-09-08  Selangor                0
6  2025-11-24  Selangor                0
7  2025-06-13  Selangor                0
8  2025-11-10  Selangor                0
9  2025-11-27  Selangor                0
IsPublicHoliday
0    660543
1     25547
Name: count, dtype: int64


### Age Group
This code converts numerical **Age** values into broader medical age groups.

- `age_bins = [-1, 17, 35, 59, 150]` defines the boundaries of each age range. The `-1` ensures that age `0` is included in the first group.
- `age_labels = ['Pediatric', 'Young Adult', 'Adult', 'Senior']` assigns category names to each age range.
- The resulting categories are stored in a new column called **Age_Group**.

| Age Range | Category |
|-----------|----------|
| 0–17 | Pediatric |
| 18–35 | Young Adult |
| 36–59 | Adult |
| 60–150 | Senior |

In [21]:
# Define the edges of your age brackets (-1 catches 0-year-old babies)
age_bins = [-1, 17, 35, 59, 150] 
age_labels = ['Pediatric', 'Young Adult', 'Adult', 'Senior']

# Create the new categorical column
df_cleaned['Age_Group'] = pd.cut(df_cleaned['Age'], bins=age_bins, labels=age_labels)

### ENCODING CATEGORICAL VARIABLES
Convert text columns to numbers using Label Encoding

In [22]:
categorical_cols = ['Name', 'State', 'District', 'NewPatient', 'StaffRole', 'StaffName', 'UpdatedGender','Age_Group', 
    'Ethnicities', 'Religions', 'Social Status', 'Countries/Geography']
encoder_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    # Ensure column is string type before encoding
    df_cleaned[col + '_Encoded'] = le.fit_transform(df_cleaned[col].astype(str))
    encoder_dict[col] = le 

### Save

In [34]:
df_cleaned.to_csv('../data/cleaned_patient_level_data.csv', index=False)

## HOURLY AGGREGATION & LAGS (For Forecasting)

In [24]:
# We group by Name and Name_Encoded to ensure the encoded ID stays with the data
# We also keep State/District info if available
group_keys = ['Name', 'Name_Encoded', 'Date', 'Hour', 'IsWeekend', 'IsPublicHoliday']

# Add State/District Encoded to grouping if they exist
if 'State_Encoded' in df_cleaned.columns: group_keys.append('State_Encoded')
if 'District_Encoded' in df_cleaned.columns: group_keys.append('District_Encoded')

# Aggregate
# hourly_demand = df_cleaned.groupby(group_keys)['PatientCount'].sum().reset_index()
# hourly_demand.rename(columns={'PatientCount': 'Calculated_Hourly_Demand'}, inplace=True)
hourly_demand = (df_cleaned.groupby(group_keys).size().reset_index(name='Calculated_Hourly_Demand'))

# --- CREATE LAGS ---
hourly_demand = hourly_demand.sort_values(['Name', 'Date', 'Hour'])

hourly_demand['Lag_24'] = hourly_demand.groupby('Name')['Calculated_Hourly_Demand'].shift(24)
hourly_demand['Lag_168'] = hourly_demand.groupby('Name')['Calculated_Hourly_Demand'].shift(168)
hourly_demand['Rolling_Mean_168'] = (
    hourly_demand.groupby('Name')['Calculated_Hourly_Demand']
    .transform(lambda x: x.shift(24).rolling(window=168, min_periods=1).mean())
)

### VALIDATION & PREVIEW

In [25]:
# 1. Check Holiday Counts (Did the logic work?)
total_holidays = df_cleaned['IsPublicHoliday'].sum()
total_rows = len(df_cleaned)
print(f"   -> Total Rows in Raw Data: {total_rows}")
print(f"   -> Total Holiday Rows: {total_holidays}")

# 2. Check Hourly Aggregation (Did columns disappear?)
print("\n   -> Preview of HOURLY DATA (Ready for Save):")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
display(hourly_demand.head(5))
print((len(hourly_demand)))

# 3. Check for NaNs (Critical for XGBoost)
nan_counts = hourly_demand[['Lag_24', 'Lag_168', 'Rolling_Mean_168']].isna().sum()
print("\n   -> NaN Check (First 7 days will be NaN due to Lags - Normal):")
print(nan_counts)

# We fill with -1 to indicate "History Not Available" (Cold Start).
# This preserves the "Young Clinics" that have < 7 days of data.
hourly_demand = hourly_demand.fillna(-1)

print(f"\n✅ NaNs handled. New Row Count: {len(hourly_demand)}")

   -> Total Rows in Raw Data: 686090
   -> Total Holiday Rows: 25547

   -> Preview of HOURLY DATA (Ready for Save):


,Name,Name_Encoded,Date,Hour,IsWeekend,IsPublicHoliday,State_Encoded,District_Encoded,Calculated_Hourly_Demand,Lag_24,Lag_168,Rolling_Mean_168
0,Clinic A,0,2025-05-01,11,0,1,11,48,3,NaN,NaN,NaN
1,Clinic A,0,2025-05-01,13,0,1,11,48,1,NaN,NaN,NaN
2,Clinic A,0,2025-05-01,17,0,1,11,48,1,NaN,NaN,NaN
3,Clinic A,0,2025-05-01,18,0,1,11,48,6,NaN,NaN,NaN
4,Clinic A,0,2025-05-02,8,0,0,11,48,1,NaN,NaN,NaN


200903

   -> NaN Check (First 7 days will be NaN due to Lags - Normal):
Lag_24               2352
Lag_168             16185
Rolling_Mean_168     2352
dtype: int64

✅ NaNs handled. New Row Count: 200903


The 'Lag_168' feature looks back 7 days (168 hours) to find patterns.
For the very first week of data in our dataset, there is no 'previous week' to look at.
Therefore, these values are undefined (NaN).

Machine Learning models (like XGBoost) cannot compute math on 'Undefined' values.
We must sacrifice the first 7 days of data to ensure the model has valid inputs for the rest.
This is standard practice in Time-Series forecasting (The 'Burn-in' period).

In [26]:
hourly_demand.isna().sum()

Name                        0
Name_Encoded                0
Date                        0
Hour                        0
IsWeekend                   0
IsPublicHoliday             0
State_Encoded               0
District_Encoded            0
Calculated_Hourly_Demand    0
Lag_24                      0
Lag_168                     0
Rolling_Mean_168            0
dtype: int64

### Save
Save the file for patient arrival modeling

In [35]:
hourly_demand.to_csv('../data/processed_hourly_demand.csv', index=False)

## Create Dataset for Staffing and Waiting Time

In [28]:
staffing_df = df_cleaned.copy()

Group by Clinic (Name) and Date.

In [29]:
golden_dataset = staffing_df.groupby(['Name', 'Date']).agg({
    'Date': 'count',                         # Total Daily Patients
    # 'PatientCount': 'sum',                 # Total Daily Patients
    'StaffName': 'nunique',               # Count distinct doctors/staff working that day
    'WaitingTime(Mins)': 'mean',          # Average Waiting Time for the day
    
    # CONTEXT (Features for the model)
    'IsWeekend': 'max',                 
    'IsPublicHoliday': 'max',          
    'State': 'first',                 
    'District': 'first'              
}).rename(columns={'Date': 'Total_Daily_Patients'}).reset_index() # Rename explicitly

# Check the result
print(golden_dataset.head())

       Name        Date  Total_Daily_Patients  StaffName  WaitingTime(Mins)  IsWeekend  IsPublicHoliday     State   District
0  Clinic A  2025-05-01                    11          2           1.516667          0                1  Selangor  Shah Alam
1  Clinic A  2025-05-02                    16          1           1.420833          0                0  Selangor  Shah Alam
2  Clinic A  2025-05-03                    20          1           0.490833          1                0  Selangor  Shah Alam
3  Clinic A  2025-05-04                    18          1           0.499074          1                0  Selangor  Shah Alam
4  Clinic A  2025-05-05                    31          2           2.830108          0                0  Selangor  Shah Alam


### ADD DERIVED FEATURES

In [30]:
# Rename
golden_dataset.rename(columns={
    'PatientCount': 'Total_Daily_Patients',
    'StaffName': 'Staff_Count',
    'WaitingTime(Mins)': 'Avg_Wait_Time'
}, inplace=True)

# Feature: Patient Load Ratio (Patients per Doctor)
golden_dataset['Patient_Load_Ratio'] = golden_dataset['Total_Daily_Patients'] / golden_dataset['Staff_Count'].replace(0, 1)

# Stress Index (The "Explosive" Queue Factor)
# Squaring the ratio captures the non-linear growth of wait times
golden_dataset['Load_Stress_Index'] = golden_dataset['Patient_Load_Ratio'] ** 2 

# Historical Baseline (Clinic Speed Score)
# This helps the model understand that "Clinic A" is naturally slower than "Clinic B"
clinic_baseline = golden_dataset.groupby('Name')['Avg_Wait_Time'].mean().rename('Historical_Base_Wait')
golden_dataset = golden_dataset.join(clinic_baseline, on='Name')

### Create Classification Target
Set "High Risk" = 1 if average wait > 30 mins, else = 0

In [31]:
golden_dataset['High_Wait_Risk'] = (golden_dataset['Avg_Wait_Time'] > 30).astype(int)

# Round decimals for cleanliness
golden_dataset['Avg_Wait_Time'] = golden_dataset['Avg_Wait_Time'].round(2)
golden_dataset['Patient_Load_Ratio'] = golden_dataset['Patient_Load_Ratio'].round(2)
golden_dataset['Load_Stress_Index'] = golden_dataset['Load_Stress_Index'].round(2)
golden_dataset['Historical_Base_Wait'] = golden_dataset['Historical_Base_Wait'].round(2)

# Fill any NaNs (rare)
golden_dataset = golden_dataset.fillna(0)

### Validation

In [32]:
golden_dataset.isna().sum()

Name                    0
Date                    0
Total_Daily_Patients    0
Staff_Count             0
Avg_Wait_Time           0
IsWeekend               0
IsPublicHoliday         0
State                   0
District                0
Patient_Load_Ratio      0
Load_Stress_Index       0
Historical_Base_Wait    0
High_Wait_Risk          0
dtype: int64

In [33]:
print("\n--- Golden Dataset Preview ---")
pd.set_option('display.max_columns', None)
display(golden_dataset.head())

print("\n--- Dataset Statistics ---")
display(golden_dataset[['Total_Daily_Patients', 'Staff_Count', 'Avg_Wait_Time']].describe())

# Check for Risk Balance
risk_counts = golden_dataset['High_Wait_Risk'].value_counts()
print(f"\n--- Risk Distribution ---\nSafe Days: {risk_counts.get(0, 0)}\nCrisis Days: {risk_counts.get(1, 0)}")


--- Golden Dataset Preview ---


,Name,Date,Total_Daily_Patients,Staff_Count,Avg_Wait_Time,IsWeekend,IsPublicHoliday,State,District,Patient_Load_Ratio,Load_Stress_Index,Historical_Base_Wait,High_Wait_Risk
0,Clinic A,2025-05-01,11,2,1.52,0,1,Selangor,Shah Alam,5.5,30.25,0.33,0
1,Clinic A,2025-05-02,16,1,1.42,0,0,Selangor,Shah Alam,16.0,256.00,0.33,0
2,Clinic A,2025-05-03,20,1,0.49,1,0,Selangor,Shah Alam,20.0,400.00,0.33,0
3,Clinic A,2025-05-04,18,1,0.50,1,0,Selangor,Shah Alam,18.0,324.00,0.33,0
4,Clinic A,2025-05-05,31,2,2.83,0,0,Selangor,Shah Alam,15.5,240.25,0.33,0



--- Dataset Statistics ---


,Total_Daily_Patients,Staff_Count,Avg_Wait_Time
count,17297.000000,17297.000000,17297.000000
mean,39.665260,2.178181,3.752339
std,30.598076,1.122082,6.731406
min,1.000000,1.000000,0.000000
25%,19.000000,1.000000,0.000000
50%,33.000000,2.000000,0.620000
75%,52.000000,3.000000,5.240000
max,234.000000,8.000000,138.600000



--- Risk Distribution ---
Safe Days: 17140
Crisis Days: 157


### Save
Save file for Staffing and Waiting Time modelling

In [36]:
# Save to CSV
golden_dataset.to_csv('../data/processed_daily_operations.csv', index=False)

# Debug

In [37]:
df_cleaned['DateTime'] = pd.to_datetime(df_cleaned['DateTime'])
clinic_open_time = df_cleaned.groupby('Name')['DateTime'].min().reset_index()
clinic_open_time = clinic_open_time.sort_values('DateTime', ascending=False)

display(clinic_open_time[clinic_open_time['Name'] == 'Clinic CJ'])

,Name,DateTime
64,Clinic CJ,2025-09-02 15:32:03
